# Fine-tune ReactionT5 on an even larger ORD sample (Model 1, Kaggle GPU)

Same idea as `01_train_reactant_ord_more_data.ipynb` (150k pool), one step further: ~250,000
ORD reactions instead of 150,000 (same seed/eval-exclusion logic -- 0 leakage verified against
`data/v2_ord_eval_targets.json`).

**Fixed after the 150k result (see `RESULTS.md` section 6):** a clean (non-augmented) 150k run
matched but did not beat v2's 60k result (ORD exact_match top-1 48.7% vs 50.7%). Root cause:
the 150k run used the script's *current* defaults (`--learning-rate` 2e-5, `--num-train-epochs`
2) instead of v2's actual proven config (`5e-5`, `3` epochs -- confirmed from v2's own saved
`training_args.bin`; the script's defaults were lowered later, for the augmentation debugging in
v3/v4, and nothing re-raised them back for the plain no-augmentation case). More data was never
cleanly tested against the *stronger* training regimen. The cell below now passes
`--learning-rate 5e-5 --num-train-epochs 3` explicitly to fix that.

**Before running:** in the notebook Settings panel (right sidebar) turn on **Internet** and
**GPU accelerator** (T4x2). Kaggle's free GPU quota is **30 hours/week**.

**Data:** built locally the same way as the 150k pool and must be uploaded as a **Kaggle
Dataset** (`reactants_train.jsonl`, `reactants_val.jsonl`), then added to this notebook as an
input (`+ Add Input`, right sidebar).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- enable GPU in Settings")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

**Input data.** Adjust the dataset slug below to match whatever you named the Kaggle Dataset you uploaded (visible under `/kaggle/input/` once added as an input).

In [ ]:
import os

train_file = "/kaggle/input/retro-planner-ord-250k/reactants_train.jsonl"  # @param {type:"string"}
val_file = "/kaggle/input/retro-planner-ord-250k/reactants_val.jsonl"  # @param {type:"string"}

assert os.path.exists(train_file), f"Not found: {train_file} -- did you add the dataset as an input (+ Add Input, right sidebar)?"
assert os.path.exists(val_file), f"Not found: {val_file}"
print("Train file:", train_file, "--", sum(1 for _ in open(train_file)), "rows")
print("Val file:", val_file, "--", sum(1 for _ in open(val_file)), "rows")

**Cross-session resume on Kaggle.** There's no Drive-style live mount here -- `/kaggle/working` only persists once you **Save Version** ("commit") the notebook, which turns its contents into this notebook's own Output, downloadable as a dataset. To continue training in a later session:

1. This session: train, then **Save Version** before your quota/time runs out. The committed `/kaggle/working/<output_dir_name>` becomes an Output you can download or directly reuse.
2. Next session: either (a) add *this same notebook's* previous Output version as an input (Kaggle lets you pick a specific version's output), or (b) download the `final`/`checkpoint-N` folder and re-upload it as its own small Dataset -- same idea as the Colab notebook's cross-account resume.
3. Point `resume_from_checkpoint_path` below at wherever that folder landed under `/kaggle/input/...`.

Leave `resume_from_checkpoint_path` blank for a first run.

In [ ]:
resume_from_checkpoint_path = ""  # @param {type:"string"}
# e.g. /kaggle/input/model1-ord250k-checkpoint/checkpoint-12500  (full Trainer checkpoint -- exact resume)
# or   /kaggle/input/model1-ord250k-checkpoint/final           (weights only -- fresh optimizer/step count)

In [ ]:
output_dir = "/kaggle/working/model1_reactant_ord250k"  # @param {type:"string"}
time_budget_minutes = 540  # @param {type:"number"}
# ~9h -- 3 epochs (matching v2's proven config, not the script's current 2-epoch default)
# over 250k/32 effective-batch is ~23,156 steps; at the ~63.5 steps/min DDP rate observed
# on the 150k Kaggle T4x2 run, that's ~6.1h, leaving headroom. Lower this for a first
# smoke-test run, or if your Kaggle session type has a shorter hard cap than this.

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"
resume_flag = ["--resume-from-checkpoint", resume_from_checkpoint_path] if resume_from_checkpoint_path else []

!python scripts/train_reactant_model_ord.py \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate 5e-5 \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(resume_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Same log-redirect reasoning as the other notebooks: printing per-step output directly in the
cell can make the tab unresponsive over a multi-hour run. Check progress by re-opening
`train.log` from the Kaggle file browser on the left, or `!tail -40 {log_path}` in a scratch cell.

**When done:** `output_dir/final` (or `output_dir/latest_checkpoint`) has the model. Evaluate it
exactly like the other checkpoints, e.g. from a local shell after downloading:

```
python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets.json \
    --t5-model <downloaded_final_dir> \
    --num-beams 10 --output experiments/v2_model1_topk/ord250k_topk.json
```